# 02 — DeMemte E5 (variante ganadora)

Reproducción end-to-end de la variante ganadora `e5_combined_dropout_ood_tau_150`:
backbone ResNet18 congelado + memoria VQ + atractor residual + gate adaptativo de 4 señales
(uncertainty, familiarity, conflict, ood_risk).

**Tres fases**:
- F1: pretrain del latente con masking (35% feature dropout)
- F2: atractor + gate + clasificadores con corrupciones aleatorias
- F3: fine-tune conjunto limpio+corrupto + anti-pareidolia loss

Con `RUN_TRAINING=False` (default) se carga el checkpoint en `out/e5_best.pt`.
El checkpoint preexistente de `experiments/atracctor/out/artifacts/dememte_e5_critical/seed_42/`
es compatible — la primera ejecución lo copia a `out/` si no encuentra uno local.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

In [ ]:
import json, shutil
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dememte.config import E5Config, resolve_data_dir
from dememte.data import build_loaders, seed_everything
from dememte.models import make_dememte_e5
from dememte.training import train_dememte_full
from dememte.evaluation import evaluate_dememte_suite, signal_curve_rows
from dememte.io import save_checkpoint, load_checkpoint, write_json, write_csv, ensure_dir

RUN_TRAINING = False

cfg = E5Config()
cfg.data_dir = resolve_data_dir(cfg)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ensure_dir(ROOT / 'notebooks' / '02_e5_winner' / 'out')
CKPT = OUT / 'e5_best.pt'
LEGACY_CKPT = ROOT / 'experiments/atracctor/out/artifacts/dememte_e5_critical/seed_42/e5_combined_dropout_ood_tau_150/e5_combined_dropout_ood_tau_150_best.pt'
seed_everything(cfg.seed)
print(json.dumps(asdict(cfg), indent=2))

## Datos

In [ ]:
tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=cfg.val_ratio,
    split_seed=cfg.split_seed,
    protocol=cfg.benchmark_protocol,
)
print(meta)

## Modelo E5

In [ ]:
model = make_dememte_e5(cfg, device=device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'params total: {n_total:,} | trainable (post-init): {n_trainable:,}')

## Entrenamiento (opcional) o carga del checkpoint

In [ ]:
if RUN_TRAINING:
    model, best_acc = train_dememte_full(model, tr_loader, va_loader, cfg, device)
    save_checkpoint(model, CKPT, extra={'best_val': best_acc, 'config': asdict(cfg)})
    print(f'saved {CKPT} (best_val={best_acc:.4f})')
else:
    if not CKPT.exists() and LEGACY_CKPT.exists():
        print('seeding out/ with legacy E5 checkpoint:', LEGACY_CKPT)
        shutil.copy(LEGACY_CKPT, CKPT)
    payload = load_checkpoint(model, CKPT, device=device, strict=True)
    print('loaded:', CKPT, '| best_val:', payload.get('best_val'))

## Evaluación clean + corrupt + métricas del gate

In [ ]:
metrics = evaluate_dememte_suite(model, te_loader, device=device, return_predictions=True)
clean_record = metrics.pop('clean_record')
corrupt_records = metrics.pop('corruption_records')

predictions = clean_record.pop('predictions', [])
for rows in corrupt_records.values():
    for rec in rows:
        predictions.extend(rec.pop('predictions', []))

summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool))}
summary.update({'protocol': meta['protocol'], 'split_seed': meta['split_seed'], 'variant': 'e5_combined_dropout_ood_tau_150'})
write_json(summary, OUT / 'metrics.json')
write_csv(predictions, OUT / 'predictions.csv')
write_csv(signal_curve_rows('e5_combined_dropout_ood_tau_150', 'E5 winner', clean_record, corrupt_records), OUT / 'signal_curves.csv')
print(json.dumps(summary, indent=2))

## Curvas del gate por corrupción/severidad

In [ ]:
curves = pd.read_csv(OUT / 'signal_curves.csv')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for corr, sub in curves[curves['corruption'] != 'clean'].groupby('corruption'):
    axes[0].plot(sub['severity'], sub['gate_mean'], marker='o', label=corr)
    axes[1].plot(sub['severity'], sub['acc'], marker='o', label=corr)
clean_row = curves[curves['corruption'] == 'clean'].iloc[0]
axes[0].axhline(clean_row['gate_mean'], linestyle='--', color='k', alpha=0.5, label='clean')
axes[1].axhline(clean_row['acc'], linestyle='--', color='k', alpha=0.5, label=f"clean ({clean_row['acc']:.3f})")
axes[0].set_xlabel('Severity'); axes[0].set_ylabel('Gate mean'); axes[0].set_title('E5 gate response')
axes[1].set_xlabel('Severity'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('E5 robustness')
for ax in axes: ax.grid(alpha=0.3); ax.legend()
ensure_dir(OUT / 'gate_plots')
fig.savefig(OUT / 'gate_plots' / 'gate_and_robustness.png', dpi=120, bbox_inches='tight')
plt.show()